# Schedulable Parameters – A More Fun Example

We optimise the 2‑D Rastrigin function with a GA while **two** parameters
change over time:
1. **Mutation scale** – a `LinearSchedule` from 0.5 to 0.01.
2. **Mutation probability** – a `StepSchedule` that starts high (0.8) for
    exploration, then drops to 0.1 after 40 % of the generations for
    exploitation.

We’ll run the algorithm manually and plot the schedules together with the
best fitness.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import metaheuristic_designer as mhd
from metaheuristic_designer.benchmarks import Rastrigin
from metaheuristic_designer.parameter_schedules import (
    LinearSchedule, StepSchedule,
)

rng = mhd.check_random_state(42)

## 1. Objective function

In [ ]:
DIM = 2
objfunc = Rastrigin(DIM, mode="min")

## 2. Schedules

- `mutation_scale`: linearly from 0.5 → 0.01
- `mutation_prob`: 0.8 until progress 0.4, then 0.1

In [ ]:
mutation_scale = LinearSchedule(init_value=0.5, final_value=0.01)
mutation_prob  = StepSchedule({0.0: 0.8, 0.4: 0.1})

# Verify at some progress values
print("Mutation scale at 0, 0.5, 1.0:", mutation_scale(0.0), mutation_scale(0.5), mutation_scale(1.0))
print("Mutation prob  at 0, 0.3, 0.5:", mutation_prob(0.0), mutation_prob(0.3), mutation_prob(0.5))

## 3. Operator with scheduled parameters

We decompose the operator into a `BranchOperator` that decides *whether* to
mutate (prob) and a composite that applies mutation (scale).

The library’s `BranchOperator` already accepts a probability as a `p` parameter.
We can replace that probability with our schedule.  Mutation scale is the `F`
parameter of the Gaussian mutation.

In [ ]:
mut_op = mhd.operators.create_operator(
    "mutation.gaussian_mutation",
    N=1,
    F=mutation_scale,          # scheduled strength
    random_state=rng,
)
# BranchOperator: with probability = mutation_prob, apply mut_op; otherwise do nothing
evolve_op = mhd.operators.BranchOperator(
    [mut_op, mhd.operators.NullOperator()],
    method="Random",
    p=mutation_prob,           # scheduled probability
    random_state=rng,
)

## 4. Fixed selections

In [ ]:
survivor_sel = mhd.survivor_selection_methods.create_survivor_selection(
    "one_to_one", random_state=rng,
)

## 5. Build strategy and algorithm

In [ ]:
strategy = mhd.SearchStrategy(
    initializer=mhd.initializers.UniformInitializer(
        DIM, -5.12, 5.12, pop_size=30, random_state=rng),
    operator=evolve_op,
    survivor_sel=survivor_sel,
    random_state=rng,
)
algo = mhd.algorithms.StandardAlgorithm(
    objfunc, strategy, ngen=100, stop_cond="ngen", verbose=False,
)

## 6. Run manually, recording schedule values

After each step, we retrieve the current parameter values from the
operators we defined earlier.

In [ ]:
scale_history, prob_history, fit_history = [], [], []
algo.initialize()
for gen in range(100):
    algo.step()
    algo.update()
    scale_history.append(mut_op.get_params()["F"])
    prob_history.append(evolve_op.get_params()["p"])
    _, best_fit = algo.best_solution()
    fit_history.append(best_fit)

## 7. Visualise everything

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True, figsize=(8, 8))

print(f"scale history: {scale_history}")
ax1.plot(scale_history, color="tab:blue")
ax1.set_ylabel("Mutation scale (F)")
ax1.grid(True)

ax2.plot(prob_history, color="tab:orange")
ax2.set_ylabel("Mutation probability")
ax2.grid(True)

ax3.plot(fit_history, color="tab:green")
ax3.set_xlabel("Generation")
ax3.set_ylabel("Best fitness")
ax3.grid(True)

plt.suptitle("Two scheduled parameters + fitness", fontsize=14)
plt.tight_layout()
plt.show()

This example shows how easy it is to combine multiple schedules.
You can replace the step schedule with a `LogisticSchedule` for smooth
transitions, or even wrap a `RandomSchedule` for stochastic exploration.